# Lab 1 — Deploy and benchmark Parakeet ASR

**Outcome:** transcribe 16 kHz speech with NVIDIA Parakeet CTC 0.6B, then measure latency, real-time factor (RTF), and throughput relative to real time. Estimated time: 75–90 minutes including discussion.

This is a single-process development deployment. Lab 3 moves the same model boundary behind NVIDIA Triton.

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'voice_asr_lab').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
os.environ.setdefault('HF_HOME', str(ROOT / '.cache' / 'huggingface'))
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

import torch
from IPython.display import Audio, display
from voice_asr_lab.audio import load_dummy_librispeech, duration_seconds
from voice_asr_lab.asr import benchmark, load_model_and_processor, transcribe
from voice_asr_lab.profiles import detect_profile

profile = detect_profile()
print(profile)

## 1. Inspect the audio contract

The model expects mono 16 kHz speech. In production, resampling, channel handling, clipping, silence, and chunking belong in an explicit preprocessing contract—not hidden inside a client.

In [ ]:
records = load_dummy_librispeech(limit=4)
sample = records[0]
print({
    'sample_rate': sample['sampling_rate'],
    'duration_seconds': round(duration_seconds(sample['audio'], sample['sampling_rate']), 2),
    'reference': sample['text'],
})
display(Audio(sample['audio'], rate=sample['sampling_rate']))

## 2. Load and transcribe

The first execution downloads roughly 2.4 GB of model weights. The helper chooses BF16 on Ampere/Ada GPUs where supported and FP16 on T4.

In [ ]:
model, processor, dtype = load_model_and_processor()
prediction = transcribe(model, processor, sample['audio'], sample['sampling_rate'])
print(f'Dtype:      {dtype}')
print(f'Reference:  {sample["text"]}')
print(f'Prediction: {prediction}')

## 3. Benchmark after warm-up

RTF is inference latency divided by audio duration; lower is better. Throughput-x-realtime is the inverse. This small test is useful for comparing settings, but a production capacity plan also needs representative duration distributions, concurrency, arrival patterns, network time, percentiles, and error budgets.

In [ ]:
metrics = benchmark(model, processor, sample['audio'], sample['sampling_rate'], repeats=5)
metrics['gpu'] = torch.cuda.get_device_name(0)
metrics['profile'] = profile.name
metrics

In [ ]:
rows = []
for index, record in enumerate(records[:3]):
    result = benchmark(model, processor, record['audio'], record['sampling_rate'], repeats=3)
    result['sample'] = index
    rows.append(result)
import pandas as pd
pd.DataFrame(rows).set_index('sample')

## Production mapping to AWS

The notebook process maps to one model replica. On EKS, preserve the same audio/model contract while adding an image, model storage, health probes, a service, GPU scheduling, metrics, autoscaling, rollout policy, and request limits. Lab 3 supplies a Triton model repository and HTTP inference boundary that can replace this in-process call.

**Checkpoint:** record your transcript, median latency, RTF, GPU and profile before moving to Lab 2.